In [1]:
import sqlite3

conn = sqlite3.connect("GTVT_law.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS laws (
    id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    content TEXT,
    parent_id TEXT,
    so_hieu TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS law_refs (
    so_hieu TEXT NOT NULL PRIMARY KEY,
    title TEXT NOT NULL,
    content TEXT
)
""")
conn.commit()

In [6]:
conn.close()

In [3]:
import win32com.client as win32
import os

def convert_doc_to_docx(input_path, output_path=None):
    word = win32.gencache.EnsureDispatch("Word.Application")
    word.Visible = False

    doc = word.Documents.Open(os.path.abspath(input_path))
    if output_path is None:
        output_path = os.path.splitext(input_path)[0] + ".docx"

    doc.SaveAs(output_path, FileFormat=16)  # 16 = wdFormatXMLDocument (.docx)
    doc.Close()
    word.Quit()

    return output_path


def batch_convert_doc_to_docx(folder="."):
    for filename in os.listdir(folder):
        if filename.lower().endswith(".doc") and not filename.lower().endswith(".docx"):
            input_path = os.path.join(folder, filename)
            print(f"Converting: {filename} ...")

            try:
                output_path = convert_doc_to_docx(input_path)
                os.remove(input_path)
                print(f"Converted and deleted: {filename}")
            except Exception as e:
                print(f"Failed to convert {filename}: {e}")

In [3]:
# Run in current folder
batch_convert_doc_to_docx(r".")

Converting: 01.2010.TTLT.BCA.BGTVT.doc ...
Converted and deleted: 01.2010.TTLT.BCA.BGTVT.doc
Converting: 09.2015.TT.BGTVT.doc ...
Converted and deleted: 09.2015.TT.BGTVT.doc
Converting: 11.2014.TT.BGTVT.doc ...
Converted and deleted: 11.2014.TT.BGTVT.doc
Converting: 13-2014-TT-BQP.doc ...
Converted and deleted: 13-2014-TT-BQP.doc
Converting: 26.2014.TTLT.BYT.BCA.doc ...
Converted and deleted: 26.2014.TTLT.BYT.BCA.doc
Converting: 27.2010.ND.CP.doc ...
Converted and deleted: 27.2010.ND.CP.doc
Converting: 32.2023.TT.BCA.doc ...
Converted and deleted: 32.2023.TT.BCA.doc
Converting: 42_2018_TT-BGTVT_395809.doc ...
Converted and deleted: 42_2018_TT-BGTVT_395809.doc
Converting: 46.2014.TT.BGTVT.doc ...
Converted and deleted: 46.2014.TT.BGTVT.doc
Converting: 49_2016_TT-BYT_326445.doc ...
Failed to convert 49_2016_TT-BYT_326445.doc: (-2147417848, 'The object invoked has disconnected from its clients.', None, None)
Converting: 53.2014.TT.BGTVT.doc ...
Converted and deleted: 53.2014.TT.BGTVT.doc


In [2]:
import re
import hashlib
from docx import Document
from docx.table import Table
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.text.paragraph import Paragraph

def extract_text_from_docx(doc_path):
    doc = Document(doc_path)
    output_text = []
    for child in doc.element.body.iterchildren():
        if isinstance(child, CT_P):
            para = Paragraph(child, doc)
            text = para.text.strip()
            if text:
                output_text.append(text)
        elif isinstance(child, CT_Tbl):
            table = Table(child, doc)
            for row in table.rows:
                for cell in row.cells:
                    text = cell.text.strip()
                    if text:
                        output_text.append(text)
    return output_text

import re

def extract_law_reference_info(text_lines):
    """
    Xử lý danh sách các dòng văn bản để trích xuất số hiệu, tựa đề và toàn bộ
    nội dung phần mở đầu (sau "Căn cứ...").
    """
    date_pattern = re.compile(r'.*ngày\s+\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}', re.IGNORECASE)

    # --- THAY ĐỔI 1: Normalize danh sách từ khóa ---
    tu_khoa_loai_bo = (
        'QUỐC HỘI',
        'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM',
        'Độc lập - Tự do - Hạnh phúc',
        '------'
    )
    # Chuyển tất cả từ khóa về chữ thường một lần duy nhất
    normalized_tu_khoa = tuple(k.lower() for k in tu_khoa_loai_bo)

    stop_keywords = ("Chương", "Điều", "Mục")
    # Tương tự, normalize cả stop_keywords
    normalized_stop_keywords = tuple(k.lower() for k in stop_keywords)

    so_hieu = None
    tua_de_parts = []
    noi_dung_parts = []

    dang_thu_thap_noi_dung = False

    for line in text_lines:
        sub_lines = line.strip().split('\n')

        for sub_line in sub_lines:
            sub_line = sub_line.strip()
            if not sub_line:
                continue

            # Normalize dòng hiện tại về chữ thường
            normalized_line = sub_line.lower()

            # --- THAY ĐỔI 2: Sử dụng các biến đã normalize để so sánh ---
            if normalized_line.startswith(normalized_stop_keywords):
                break

            # Bật chế độ thu thập nội dung khi gặp "Căn cứ" (không phân biệt hoa/thường)
            if normalized_line.startswith('căn cứ'):
                dang_thu_thap_noi_dung = True

            if dang_thu_thap_noi_dung:
                noi_dung_parts.append(sub_line)
                continue

            match_sh = re.search(r'Số:\s*(.+)', sub_line, re.IGNORECASE)
            if match_sh:
                so_hieu = match_sh.group(1).replace(' ', '')
                continue

            # So sánh dòng đã normalize với từ khóa đã normalize
            if normalized_line.startswith(normalized_tu_khoa) or date_pattern.match(normalized_line):
                continue

            tua_de_parts.append(sub_line)

        if line.strip().lower().startswith(normalized_stop_keywords):
            break

    final_title = ' '.join(tua_de_parts).strip()
    final_content = ' '.join(noi_dung_parts).strip()

    return so_hieu, final_title, final_content

def create_law_ref(cursor, connection, so_hieu, title, content):
    cursor.execute(
        """
        REPLACE INTO law_refs (so_hieu, title, content)
        VALUES (?, ?, ?)
        """, (so_hieu, title, content)
    )
    connection.commit()

def collect_content(output_text, start_index):
    content_lines = []
    i = start_index
    while i < len(output_text):
        next_line = output_text[i].strip()
        if (next_line.startswith("Điều")
            or next_line.startswith("Chương")
            or next_line.startswith("Mục")
            or re.match(r"^\s*(\d+\.\s*|[a-zA-ZđĐ]\)\s*)", next_line)):
            break
        if next_line:
            content_lines.append(next_line)
        i += 1
    return "\n".join(content_lines).strip() or None, i

def clean_content(text):
    if text:
        return " ".join(text.split())
    return None

def generate_id(full_path_title):
    return hashlib.sha256(full_path_title.encode('utf-8')).hexdigest()

def create_law_entry(title, content, parent_id, so_hieu, full_path_title, cursor, connection):
    """
    Tạo ID bằng cách băm full_path_title và chèn vào DB.
    Lưu ý: Bảng 'laws' bây giờ phải có cột 'id' kiểu TEXT làm khóa chính.
    """
    entry_id = generate_id(full_path_title)
    cursor.execute(
        "REPLACE INTO laws (id, title, content, parent_id, so_hieu) VALUES (?, ?, ?, ?, ?)",
        (entry_id, title, content, parent_id, so_hieu)
    )
    connection.commit()
    return entry_id

def parse_document(output_text, cursor, conn):
    index = 0
    so_hieu, law_title, law_content = extract_law_reference_info(output_text)
    create_law_ref(cursor, conn, so_hieu, law_title, law_content)
    if not so_hieu:
        raise ValueError("Không tìm thấy số hiệu văn bản.")

    # Biến để lưu trữ ID và tiêu đề của các mục cha hiện tại
    chuong_id = muc_id = dieu_id = khoan_id = None
    chuong_title = muc_title = dieu_title = khoan_title = ""

    while index < len(output_text):
        line = output_text[index].strip()

        # ---- Chương ----
        if line.startswith("Chương"):
            parts = line.split("\n", 1)
            title = parts[0].strip()
            content, index = collect_content(output_text, index + 1)
            content = clean_content(content)

            full_path = f"{so_hieu}_{title}"
            chuong_id = create_law_entry(title, content, None, so_hieu, full_path, cursor, conn)

            # Cập nhật đường dẫn hiện tại và reset các cấp con
            chuong_title = title
            muc_id = dieu_id = khoan_id = None
            muc_title = dieu_title = khoan_title = ""
            continue

        # ---- Mục ----
        if line.startswith("Mục"):
            parts = line.split("\n", 1)
            title = parts[0].strip()
            content, index = collect_content(output_text, index + 1)
            content = clean_content(content)

            full_path = f"{so_hieu}_{chuong_title}_{title}"
            muc_id = create_law_entry(title, content, chuong_id, so_hieu, full_path, cursor, conn)

            # Cập nhật đường dẫn hiện tại và reset các cấp con
            muc_title = title
            dieu_id = khoan_id = None
            dieu_title = khoan_title = ""
            continue

        # ---- Điều ----
        if line.startswith("Điều"):
            parts = re.split(r"\.", line, maxsplit=1)
            title = parts[0].strip()
            content = parts[1].strip() if len(parts) > 1 else ""
            extra_content, index = collect_content(output_text, index + 1)
            if extra_content:
                content = f"{content}\n{extra_content}" if content else extra_content
            content = clean_content(content)

            parent_id = muc_id if muc_id else chuong_id
            parent_path = f"{so_hieu}_{chuong_title}"
            if muc_title:
                parent_path += f"_{muc_title}"

            full_path = f"{parent_path}_{title}"
            dieu_id = create_law_entry(title, content, parent_id, so_hieu, full_path, cursor, conn)

            # Cập nhật đường dẫn và reset cấp con
            dieu_title = title
            khoan_id = None
            khoan_title = ""
            continue

        # ---- Khoản ----
        if re.match(r"^\d+\.", line):
            parts = re.split(r"\.", line, maxsplit=1)
            title_num = parts[0].strip()
            title = f"Khoản {title_num}"
            content = parts[1].strip() if len(parts) > 1 else ""
            extra_content, index = collect_content(output_text, index + 1)
            if extra_content:
                content = f"{content}\n{extra_content}" if content else extra_content
            content = clean_content(content)

            parent_path = f"{so_hieu}_{chuong_title}"
            if muc_title: parent_path += f"_{muc_title}"
            parent_path += f"_{dieu_title}"

            full_path = f"{parent_path}_{title}"
            khoan_id = create_law_entry(title, content, dieu_id, so_hieu, full_path, cursor, conn)
            khoan_title = title
            continue

        # ---- Điểm ----
        match = re.match(r"^([^\W\d_])\)", line, re.UNICODE)
        if match:
            letter = match.group(1)
            title = f"Điểm {letter}"
            parts = re.split(r"\)", line, maxsplit=1)
            content = parts[1].strip() if len(parts) > 1 else ""
            extra_content, index = collect_content(output_text, index + 1)
            if extra_content:
                content = f"{content}\n{extra_content}" if content else extra_content
            content = clean_content(content)

            parent_path = f"{so_hieu}_{chuong_title}"
            if muc_title: parent_path += f"_{muc_title}"
            parent_path += f"_{dieu_title}_{khoan_title}"

            full_path = f"{parent_path}_{title}"
            create_law_entry(title, content, khoan_id, so_hieu, full_path, cursor, conn)
            continue

        index += 1

In [22]:
cursor.execute("DELETE FROM laws")
cursor.execute("DELETE FROM law_refs")
conn.commit()

In [24]:
# "Luật-46-2014-QH13.docx" còn nhiều ngoại lệ
doc_path = r"./TT.41.2021.TT.BGTVT.docx"

output_text = extract_text_from_docx(doc_path)
print(output_text)

['BỘ\xa0GIAO THÔNG VẬN TẢI\n__________', 'CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM\nĐộc lập - Tự do - Hạnh phúc', 'Số: 41/2021/TT-BGTVT', 'Hà\xa0Nội, ngày\xa031 tháng 12\xa0năm\xa02021', 'THÔNG TƯ', 'Sửa đổi, bổ sung một số điều của Thông tư số 37/2018/TT-BGTVT ngày 07 tháng 6 năm 2018 của Bộ trưởng Bộ Giao thông vận tải quy định về quản lý, vận hành khai thác và bảo trì công trình đường bộ', 'Căn cứ Luật Giao thông đường bộ số\xa023/2008/QH12 ngày 13 tháng 11 năm 2008;', 'Căn cứ Nghị định\xa012/2017/NĐ-CP ngày 10 tháng 02 năm 2017 của Chính phủ quy định chức năng, nhiệm\xa0vụ, quyền hạn và\xa0cơ cấu tổ chức của Bộ Giao thông vận tải;', 'Căn cứ Nghị định số 06/2021/NĐ-CP ngày\xa026 tháng 01\xa0năm\xa02021 của\xa0Chính phủ\xa0về\xa0quản lý chất lượng, thi công xây dựng  và\xa0bảo trì\xa0công trình xây dựng;', 'Căn cứ Nghị định số\xa032/2014/NĐ-CP\xa0ngày 22 tháng 4 năm 2014 của Chính phủ quy định về\xa0quản\xa0lý, khai thác và\xa0bảo\xa0trì đường cao\xa0tốc;', 'Căn cứ Nghị định số 33/2019/NĐ-

In [4]:
import os

doc_path = r"."

for filename in os.listdir(doc_path):
    if filename.endswith(".docx"):
        try:
            output_text = extract_text_from_docx(os.path.join(doc_path, filename))
            parse_document(output_text, cursor, conn)
            print(f"Da xu ly thanh cong: {filename}")

        except Exception as e:
            print(f"Loi khi xu ly file {filename}: {str(e)}")
            continue

Da xu ly thanh cong: 37_2018_TT-BGTVT.docx
Da xu ly thanh cong: luattrậttự.docx
Da xu ly thanh cong: TT.16.2022.TT.B GTVT.docx
Da xu ly thanh cong: TT.34.2021.TT.BGTVT.docx
Da xu ly thanh cong: TT.36.2020.TT.BGTVT.docx
Da xu ly thanh cong: TT.40.2021.TT.BGTVT.docx
Da xu ly thanh cong: TT.41.2021.TT.BGTVT.docx
